# singular value decomposition

In [ ]:
import torch
import numpy as np
_ = torch.manual_seed(0)

generate a rank deficient martrix w

In [ ]:
d, k = 10, 10

# this way we can generate a rank-deficient matrix
W_rank = 2
W = torch.randn(d, W_rank) @ torch.randn(W_rank, k)
print(W)

evaluate the rank of the matrix W

In [ ]:
W_rank = np.linalg.matrix_rank(W)
print(f'rank of W: {W_rank}')

calculate the svd decomposition of the W matrix

In [ ]:
# perform svd on W (W = UxSxV^T)
U, S, V = torch.svd(W)


# for rank-r factorization, keep only the first r singular values (and corresponding columns of U and V)
U_r = U[:, :W_rank]
S_r = torch.diag(S[:W_rank])
V_r = V[:, :W_rank].t()


# compute C = U_r * S_r and R = V_r
B = U_r @ S_r
A = V_r
print(f'shape of B: {B.shape}')
print(f'shape of A: {A.shape}')

given the same input, check the output using the original W matrix and the matrices resulting from the decomposition

In [ ]:
# generate random bias and input
bias = torch.randn(d)
x = torch.randn(d)

# compute y = Wx + b
y = W @ x + bias

# compute y' = CRx + b
y_prime = (B @ A) @ x + bias

print("original y using W:\n", y)
print("")
print("y' computed using BA:\n", y_prime)

In [ ]:
print("total parameters of W: ", W.nelement())
print("total parameters of B and A: ", B.nelement() + A.nelement())


# lora implementation with pytorch

let's start by importing the necessary libraries

In [ ]:
import torch
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm import tqdm

make the model deterministic

In [ ]:
# make torch deterministic
_ = torch.manual_seed(0)

we will be training a network to classify MNIST digits and the fine-tune the network on a particular digit on which it doesn't perform well.

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])

# load the mnist dataset
mnist_trainset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
# create a dataloader for the training
train_loader = torch.utils.data.DataLoader(mnist_trainset, batch_size=10, shuffle=True)

# load the mnist test set
mnist_testset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(mnist_testset, batch_size=10, shuffle=True)

# define the device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

create the neural network to classify the digits, making it overly complicated to better show the power of LoRA

In [ ]:
# create an overly expensive neural network to classify MNIST digits
# daddy got money, so i don't care about efficiency

class RichBoyNet(nn.Module):
    def __init__(self, hidden_size_1=1000, hidden_size_2=2000):
        super(RichBoyNet, self).__init__()
        self.linear1 = nn.Linear(28*28, hidden_size_1)
        self.linear2 = nn.Linear(hidden_size_1, hidden_size_2)
        self.linear3 = nn.Linear(hidden_size_2, 10)
        self.relu = nn.ReLU()

    def forward(self, img):
        x = img.view(-1, 28*28)
        x = self.relu(self.linear1(x))
        x = self.relu(self.linear2(x))
        x = self.linear3(x)
        return x

net = RichBoyNet().to(device)

train the network only for 1 epoch to simulate a complete general pre-training on the data

In [ ]:
def train(train_loader, net, epochs=5, total_iterations_limit=None):
    cross_el = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=0.001)

    total_iterations = 0

    for epoch in range(epochs):
        net.train()

        loss_sum = 0
        num_iterations = 0

        data_iterator = tqdm(train_loader, desc=f'epoch {epoch+1}')

        if total_iterations_limit is not None:
            data_iterator.total = total_iterations_limit
        for data in data_iterator:
            num_iterations += 1
            total_iterations += 1
            x, y = data
            x = x.to(device)
            y = y.to(device)
            optimizer.zero_grad()
            output = net(x.view(-1, 28*28))
            loss = cross_el(output, y)
            loss_sum += loss.item()
            avg_loss = loss_sum / num_iterations
            data_iterator.set_postfix(loss=avg_loss)
            loss.backward()
            optimizer.step()

            if total_iterations_limit is not None and total_iterations >= total_iterations_limit:
                return

train(train_loader, net, epochs=1)

keep a copy of the original weights (cloning them) so later we can prove that a fine-tuning with LoRA doesn't alter the original weights

In [ ]:
original_weights = {}
for name, param in net.named_parameters():
    original_weights[name] = param.clone().detach()

the performance of the pretrained network. as we can see, the network performs poorly on the digit 9. let's fine-tune it on the digit 9.

In [ ]:
def test():
    correct = 0
    total = 0

    wrong_counts = [0 for i in range(10)]

    with torch.no_grad():
        for data in tqdm(test_loader, desc='Testing'):
            x, y = data
            x = x.to(device)
            y = y.to(device)
            output = net(x.view(-1, 784))
            for idx, i in enumerate(output):
                if torch.argmax(i) == y[idx]:
                    correct += 1
                else:
                    wrong_counts[y[idx]] += 1
                total += 1
    print(f'accuracy: {round(correct/total, 3)}')
    for i in range(len(wrong_counts)):
        print(f'wrong counts for the digit {i}: {wrong_counts[i]}')

test()

let's visualize how many parameters are in the original network, before introducing the LoRA matrices.

In [ ]:
# print the size of the weights matrices of the network
# save the count of the total number of parameters

total_parameters_original = 0
for index, layer in enumerate([net.linear1, net.linear2, net.linear3]):
    total_parameters_original += layer.weight.nelement() + layer.bias.nelement()
    print(f'layer {index+1}: W: {layer.weight.shape} + B: {layer.bias.shape}')
print(f'total number of parameters: {total_parameters_original:,}')

define the lora parameterization as described in the paper.

In [ ]:
class LoRAParametrization(nn.Module):
    def __init__(self, features_in, features_out, rank=1, alpha=1, device="cpu"):
        super().__init__()
        # section 4.1 of the paper:
        # we use a random gaussian initialization for A and zero for B, so dW = BA is zero at the beginning of training
        self.lora_A = nn.Parameter(torch.zeros((rank, features_out)).to(device))
        self.lora_B = nn.Parameter(torch.zeros((features_in, rank)).to(device))
        nn.init.normal_(self.lora_A, mean=0, std=1)

        self.scale = alpha / rank
        self.enabled = True


    def forward(self, original_weights):
        if self.enabled:
            # return X + (B*A)*scale
            return original_weights + torch.matmul(self.lora_B, self.lora_A).view(original_weights.shape) * self.scale

        else:
            return original_weights

add the parameterization to our network

In [ ]:
import torch.nn.utils.parametrize as parametrize

def linear_layer_parameterization(layer, device, rank=1, lora_alpha=1):
    # only add the parameterization on the weight matrix, ignore the bias

    features_in, features_out = layer.weight.shape
    return LoRAParametrization(
        features_in, features_out, rank=rank, alpha=lora_alpha, device=device
    )

In [ ]:
parametrize.register_parametrization(
    net.linear1, "weight", linear_layer_parameterization(net.linear1, device)
)

parametrize.register_parametrization(
    net.linear2, "weight", linear_layer_parameterization(net.linear2, device)
)

parametrize.register_parametrization(
    net.linear3, "weight", linear_layer_parameterization(net.linear3, device)
)


In [ ]:
def enable_disable_lora(enabled=True):
    for layer in [net.linear1, net.linear2, net.linear3]:
        layer.parametrizations["weight"][0].enabled = enabled

display the number of parameters added by LoRA

In [ ]:
total_parameters_lora = 0
total_parameters_non_lora = 0

for index, layer in enumerate([net.linear1, net.linear2, net.linear3]):
    total_parameters_lora += layer.parametrizations["weight"][0].lora_A.nelement() + layer.parametrizations["weight"][0].lora_B.nelement()
    total_parameters_non_lora += layer.weight.nelement() + layer.bias.nelement()
    print(
        f'layer {index+1}: W: {layer.weight.shape} + B: {layer.bias.shape} + LoRA_A: {layer.parametrizations["weight"][0].lora_A.shape} + LoRA_B: {layer.parametrizations["weight"][0].lora_B.shape}'
    )

# the non-LoRA parameters count must match the original network
assert total_parameters_non_lora == total_parameters_original
print(f'total number of parameters (original): {total_parameters_non_lora:,}')
print(f'total number of parameters (original + LoRA): {total_parameters_lora + total_parameters_non_lora:,}')
print(f'parameters introduced by LoRA: {total_parameters_lora:,}')
parameters_increment = (total_parameters_lora / total_parameters_non_lora) * 100
print(f'parameters increment: {parameters_increment:.3f}%')

freeze all the parameters of the original network and only fine tuning the ones introduced by LoRA. then fine-tune the model on the digit 9 and only for 100 batches.

In [ ]:
# freeze the non-lora parameters
for name, param in net.named_parameters():
    if 'lora' not in name:
        print(f'freezing non-LoRA parameter {name}')
        param.requires_grad = False

# load the MNIST dataset again, by keeping only the digit 9
mnist_trainset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
exclude_indices = mnist_trainset.targets == 9
mnist_trainset.data = mnist_trainset.data[exclude_indices]
mnist_trainset.targets = mnist_trainset.targets[exclude_indices]

# create a dataloader for the training
train_loader = torch.utils.data.DataLoader(mnist_trainset, batch_size=10, shuffle=True)

# train the network with LoRA only on the digit 9 and only for 100 batches (hoping that it would improve the performance on the digit 9)
train(train_loader, net, epochs=1, total_iterations_limit=100)

verify that the fine-tuning didn't alter the original weights, but only the ones introduced by LoRA

In [ ]:
# check that the frozen parameters are still unchanged by the finetuning
assert torch.all(net.linear1.parametrizations.weight.original == original_weights['linear1.weight'])
assert torch.all(net.linear2.parametrizations.weight.original == original_weights['linear2.weight'])
assert torch.all(net.linear3.parametrizations.weight.original == original_weights['linear3.weight'])

enable_disable_lora(enabled=True)
# the new linear1.weight is obtained by the "forward" function of our LoRA parametrization
# the original weights have been moved to net.linear1.parametrizations.weight.original
assert torch.equal(net.linear1.weight, net.linear1.parametrizations.weight.original + (net.linear1.parametrizations.weight[0].lora_B @ net.linear1.parametrizations.weight[0].lora_A) * net.linear1.parametrizations.weight[0].scale)

enable_disable_lora(enabled=False)
# if we disable LoRA, the linear1.weight is the original one
assert torch.equal(net.linear1.weight, original_weights['linear1.weight'])

test the network with LoRA enabled (the digit 9 should be classified better)

In [ ]:
# test with LoRA enabled
enable_disable_lora(enabled=True)
test()



test the network with LoRA disabled (the accuracy and errors counts must be the same as the original network)

In [ ]:
# test with LoRA disabled
enable_disable_lora(enabled=False)
test()